README

This is the notebook submission for assignment 3.

For categorical values, missing data is randomly imputed, since the distribution is uniform irrespective of class. One-hot encoding is not used firstly because SMOTE can not handle such data, and also because XGBoost can natively handle it.
For numeric values, mean imputation is done followed by standard (z-score) scaling.

SMOTE is used to generate synthetic data samples from the positive class, to correct the imbalance such that the new dataset has about 20 % positive classes (rather than initial 7 %).

k-fold Cross Validation is used, with k = 5.

The model is XGBoost. For the submission "submission_19Oct_7.csv", the model is XGBoost with 150 base learners, learning rate = 0.075 and scale_pos_weight=1.5 (due to the imbalance). 

In [1]:
import numpy as np
import pandas as pd 

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin, clone
from xgboost import XGBClassifier
from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay
from imblearn.over_sampling import SMOTENC

In [2]:
train_df = pd.read_csv("train.csv")
string_columns = ["tgfrqg", "m6gvdp", "jjp9hi"]
numeric_columns = train_df.select_dtypes(include="number").columns
numeric_columns = numeric_columns[1:-1] # Remove the ID and label columns
all_columns = string_columns + numeric_columns.to_list()

In [3]:
# Part 1 : Missing Values, Standardization, Synthetic Data

class RandomChoiceImputer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        # X is the sub-DataFrame for whom we want to do the imputation
        # print(X.T)
        self.columnwise_distinct_strings_ = {}
        for col in X:
            column = X[col].to_numpy().T
            column = [x for x in column if isinstance(x, str)]
            distinct_strings = np.unique(column)
            self.columnwise_distinct_strings_[col] = distinct_strings.tolist()
        return self

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

    def transform(self, X, y=None):
        X_transformed = X.copy()
        for col in X_transformed:
            mask = X_transformed[col].isna()
            X_transformed.loc[mask, col] = np.random.choice(self.columnwise_distinct_strings_[col], size = mask.sum())

        # print(X_transformed.T)
        return X_transformed

numeric_preprocessor= Pipeline(
    steps=[
        ("imputation_mean", SimpleImputer(missing_values=np.nan, strategy="mean")),
        ("scaler", StandardScaler())
    ]
)

string_preprocessor = Pipeline(
    steps = [
        ("Imputation", RandomChoiceImputer()) # No need for one-hot encoding
    ]
)

preprocessor = ColumnTransformer(
    [
        ("string", string_preprocessor, string_columns),
        ("numeric", numeric_preprocessor, numeric_columns)
    ]
)

smote_nc = SMOTENC(categorical_features="auto", random_state=42, sampling_strategy=0.25)

In [4]:
# Part 2 : Training and Validation

train_y = train_df["serious_delinquency"].copy()
k = 5
step_size = int(len(train_df) / k)
num_rows = len(train_df)
train_df_partitions = []
for i in range(k):
    partition = []
    partition_validation = train_df[i * step_size : min((i + 1) * step_size, num_rows)]
    partition_validation_y = train_y[i * step_size : min((i + 1) * step_size, num_rows)]
    partition_train = train_df.iloc[np.r_[0:(i * step_size), min((i + 1) * step_size, num_rows):num_rows]]
    partition_train_y = train_y[np.r_[0:(i * step_size), min((i + 1) * step_size, num_rows):num_rows]]
    partition.extend([partition_validation, partition_validation_y, partition_train, partition_train_y])
    train_df_partitions.append(partition)

    print("There are", partition_train_y.sum(), "positive classes in training set", i)

for i in range(k):
    new_preprocessor = clone(preprocessor)
    new_smote_nc = clone(smote_nc)

    partition = train_df_partitions[i]
    processed_train_data = new_preprocessor.fit_transform(partition[2])
    processed_train_data = pd.DataFrame(processed_train_data, columns=all_columns)
    processed_train_data[string_columns] = processed_train_data[string_columns].astype("category")
    processed_train_data[numeric_columns] = processed_train_data[numeric_columns].astype("float64")
    processed_train_data, processed_train_y = smote_nc.fit_resample(processed_train_data, partition[3])

    model = XGBClassifier(n_estimators = 100, learning_rate=0.1, scale_pos_weight=1.5, random_state=42)
    processed_validation_data = new_preprocessor.transform(partition[0])
    processed_validation_data = pd.DataFrame(processed_validation_data, columns=all_columns)
    processed_validation_data[string_columns] = processed_validation_data[string_columns].astype("category")
    processed_validation_data[numeric_columns] = processed_validation_data[numeric_columns].astype("float64")
    model.fit(processed_train_data, processed_train_y)
    y_probs = model.predict_proba(processed_validation_data)[:, 1]
    auc_score = roc_auc_score(partition[1], y_probs)
    print("The AUC score is", auc_score)


There are 5614 positive classes in training set 0
There are 5616 positive classes in training set 1
There are 5616 positive classes in training set 2
There are 5608 positive classes in training set 3
There are 5614 positive classes in training set 4
The AUC score is 0.8431592201088434
The AUC score is 0.8475362603571166
The AUC score is 0.8521777703669701
The AUC score is 0.8409411344941239
The AUC score is 0.84007697208457


In [7]:
# Part 3 : Submission

new_preprocessor = clone(preprocessor)
new_smote_nc = clone(smote_nc)

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
test_df_id_col = test_df["id"].copy()
test_df = test_df.drop(columns=["id"])

train_y = train_df["serious_delinquency"].copy()
train_df = new_preprocessor.fit_transform(train_df)
train_df = pd.DataFrame(train_df, columns=all_columns)
train_df[string_columns] = train_df[string_columns].astype("category")
train_df[numeric_columns] = train_df[numeric_columns].astype("float64")
print("len(train_df) =", len(train_df))
processed_train_df, processed_train_y = new_smote_nc.fit_resample(train_df, train_y)

test_df = new_preprocessor.transform(test_df)
test_df = pd.DataFrame(test_df, columns=all_columns)
test_df[string_columns] = test_df[string_columns].astype("category")
test_df[numeric_columns] = test_df[numeric_columns].astype("float64")

model = XGBClassifier(n_estimators=150, learning_rate=0.075, scale_pos_weight=1.5, random_state=42)
model.fit(processed_train_df, processed_train_y)
y_probs = model.predict_proba(test_df)[:, 1]

final_answer = pd.concat([test_df_id_col, pd.Series(y_probs)], axis=1)
final_answer.columns = ["id", "default_probability"]
print(final_answer.info())
final_answer.to_csv("submission_19Oct_7.csv", index=False)

len(train_df) = 104990
<class 'pandas.DataFrame'>
RangeIndex: 22498 entries, 0 to 22497
Data columns (total 2 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   id                   22498 non-null  int64  
 1   default_probability  22498 non-null  float32
dtypes: float32(1), int64(1)
memory usage: 263.8 KB
None
